In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shahriar26s/banana-ripeness-classification-dataset")

print("Path to dataset files:", path)

100%|██████████| 221M/221M [00:01<00:00, 132MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1


In [ ]:
import os
for root, dirs, files in os.walk(path):
    print(root, len(files), "files")

/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1 0 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset 0 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset/test 0 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset/test/overripe 113 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset/test/rotten 185 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset/test/ripe 154 files
/root/.cache/kagglehub/datasets/shahriar26s/banana-ripeness-classification-dataset/versions/1/Banana Ripeness Classification Dataset/test/unripe 1

In [ ]:
for root, dirs, files in os.walk(path):
    if files:
        folder_name = os.path.relpath(root, path)
        print(folder_name, "->", len(files), "files")

Banana Ripeness Classification Dataset/test/overripe -> 113 files
Banana Ripeness Classification Dataset/test/rotten -> 185 files
Banana Ripeness Classification Dataset/test/ripe -> 154 files
Banana Ripeness Classification Dataset/test/unripe -> 110 files
Banana Ripeness Classification Dataset/valid/overripe -> 229 files
Banana Ripeness Classification Dataset/valid/rotten -> 388 files
Banana Ripeness Classification Dataset/valid/ripe -> 339 files
Banana Ripeness Classification Dataset/valid/unripe -> 167 files
Banana Ripeness Classification Dataset/train/overripe -> 2349 files
Banana Ripeness Classification Dataset/train/rotten -> 4020 files
Banana Ripeness Classification Dataset/train/ripe -> 3522 files
Banana Ripeness Classification Dataset/train/unripe -> 1902 files


In [ ]:
import shutil

dataset_root = os.path.join(path, "Banana Ripeness Classification Dataset")
clean_base = "/content/dataset"
splits = ["train", "valid", "test"]
classes = {"unripe": "green", "ripe": "yellow"}

for split in splits:
    for src_class, dst_class in classes.items():
        src_dir = os.path.join(dataset_root, split, src_class)
        dst_dir = os.path.join(clean_base, split, dst_class)
        os.makedirs(dst_dir, exist_ok=True)
        for f in os.listdir(src_dir):
            shutil.copy(os.path.join(src_dir, f), os.path.join(dst_dir, f))

print("Done — clean dataset ready at:", clean_base)

Done — clean dataset ready at: /content/dataset


In [ ]:
import tensorflow as tf

IMAGE_HEIGHT = 128
IMAGE_WIDTH = 128
BATCH_SIZE = 32
SEED = 42

train_dir = "/content/dataset/train"
val_dir = "/content/dataset/valid"
test_dir = "/content/dataset/test"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE, seed=SEED, shuffle=True,
    label_mode='binary'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE, seed=SEED, shuffle=False,
    label_mode='binary'
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(IMAGE_HEIGHT, IMAGE_WIDTH),
    batch_size=BATCH_SIZE, seed=SEED, shuffle=False,
    label_mode='binary'
)

class_names = train_dataset.class_names
print("Classes:", class_names)

Found 5424 files belonging to 2 classes.
Found 506 files belonging to 2 classes.
Found 264 files belonging to 2 classes.
Classes: ['green', 'yellow']


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
], name='data_augmentation')

def build_transfer_model(input_shape, augmentation):
    base_model = tf.keras.applications.MobileNetV3Small(
        weights='imagenet', input_shape=input_shape, include_top=False
    )
    preprocess_fn = tf.keras.applications.mobilenet_v3.preprocess_input
    base_model.trainable = False  # Phase 1: freeze everything first

    inputs = tf.keras.Input(shape=input_shape)
    x = augmentation(inputs)
    x = preprocess_fn(x)
    x = base_model(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
    return tf.keras.Model(inputs, outputs, name='banana_tl_model'), base_model

INPUT_SHAPE = (IMAGE_HEIGHT, IMAGE_WIDTH, 3)
model, base_model = build_transfer_model(INPUT_SHAPE, data_augmentation)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/applications/mobilenet_v3.py:454: UserWarning: `input_shape` is undefined or non-square, or `rows` is not 224. Weights for input shape (224, 224) will be loaded as the default.
  return MobileNetV3(


4334752/4334752 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "banana_tl_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ data_augmentation (Sequential)  │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ MobileNetV3Small (Functional)   │ (None, 4, 4, 576)      │       939,120 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 576)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       147,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           257 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,087,089 (4.15 MB)

 Trainable params: 147,969 (578.00 KB)

 Non-trainable params: 939,120 (3.58 MB)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath='models/banana_best.keras',
        monitor='val_accuracy', save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=5, restore_best_weights=True, verbose=1
    ),
]

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 64s 304ms/step - accuracy: 0.8966 - loss: 0.2688 - val_accuracy: 1.0000 - val_loss: 0.0488
Epoch 2/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 48s 285ms/step - accuracy: 0.9919 - loss: 0.0544 - val_accuracy: 1.0000 - val_loss: 0.0167
Epoch 3/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 49s 286ms/step - accuracy: 0.9961 - loss: 0.0274 - val_accuracy: 1.0000 - val_loss: 0.0084
Epoch 4/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 48s 282ms/step - accuracy: 0.9965 - loss: 0.0190 - val_accuracy: 1.0000 - val_loss: 0.0051
Epoch 5/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 83s 287ms/step - accuracy: 0.9978 - loss: 0.0151 - val_accuracy: 1.0000 - val_loss: 0.0039
Epoch 6/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 81s 281ms/step - accuracy: 0.9987 - loss: 0.0109 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 7/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 49s 288ms/step - accuracy: 0.9983 - loss: 0.0097 - val_accuracy: 1.0000 - val_loss: 0.0021
Epoch 8/15
170/170 ━━━━━━━━━━━━━━━━━━━━ 47s 279ms/step - accuracy: 0.9987 - loss: 0

In [ ]:
test_loss, test_acc = model.evaluate(test_dataset, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

model.save('models/banana_model.keras')
print("Model saved to models/banana_model.keras")

Test Accuracy: 1.0000
Test Loss: 0.0007
Model saved to models/banana_model.keras


In [ ]:
model.save('models/banana_model.keras')

from google.colab import files
files.download('models/banana_model.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>